# QLoRA Fine-tuning: TinyLlama на маркетинговом датасете

Здесь я дообучаю TinyLlama-1.1B под стиль маркетингового копирайтера.
Выбрала TinyLlama а не Mistral-7B потому что T4 имеет 15GB VRAM
Mistral при batch_size>2 стабильно падал по памяти на тестовых запусках.
TinyLlama обучается быстрее и предсказуемее на бесплатном Colab

## Шаг 0 - Проверка GPU

Первым делом проверила что GPU подключился

In [1]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Fri Jun 19 12:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Установка зависимостей

Фиксирую версии, иначе transformers и peft иногда конфликтуют.
Bitsandbytes нужен именно для 4-bit квантизации, без него QLoRA не работает.

In [2]:
!pip install -q transformers==4.45.0 trl==0.11.0 --force-reinstall

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.12.1 which is incompatible.
libcuml-cu12 26.2.0 

## Шаг 2 - Загрузка датасета

Датасет собирала из yahma/alpaca-cleaned, отфильтровала маркетинговые примеры
по ключевым словам + вручную проверяла и удаляла ~380 записей.
Итого 640 чистых примеров в формате instruction-response.

In [3]:
from google.colab import files
uploaded = files.upload()  # загружаем marketing_dataset.jsonl

Saving marketing_dataset (4).jsonl to marketing_dataset (4) (5).jsonl


In [4]:
import json
from datasets import Dataset

data = []
with open('marketing_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f'Загружено: {len(data)} примеров')
print('Первый пример:')
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Загружено: 314 примеров
Первый пример:
{
  "instruction": "Generate a list of marketing strategies to promote a new mobile app.",
  "response": "Here are some marketing strategies to promote a new mobile app:\n\n1. Social media advertising campaigns: Utilize major social media platforms such as Facebook, Instagram and Twitter to reach a large audience and promote the app.\n\n2. App Store Optimization: Optimize the app's title, description, and keywords to rank higher in app store search results.\n\n3. Influencer marketing: Partner with social media influencers and bloggers to spread the word and increase visibility of the app.\n\n4. Content marketing: Develop useful and informative blog posts, videos, and infographics to attract and engage potential customers.\n\n5. Email marketing: Create targeted email campaigns to promote the app to a specific audience.\n\n6. Referral marketing: Encourage current users to refer their friends and family members to the app in exchange for rewards or i

In [5]:
# TinyLlama использует chat-template с тегами <|system|>, <|user|>, <|assistant|> поэтому добавила системный промпт который задаёт стиль иначе модель
# будет отвечать слишком обобщённо
def format_prompt(example):
    return {
        'text': (
            f"<|system|>You are a professional marketing copywriter. "
            f"Write clear, concise, and compelling marketing content.</s>"
            f"<|user|>{example['instruction']}</s>"
            f"<|assistant|>{example['response']}</s>"
        )
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)

# 90/10 split, val нужен чтобы следить за переобучением
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(dataset["train"])} | Val: {len(dataset["test"])}')
print('\nПример форматированной строки:')
print(dataset['train'][0]['text'][:400])

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Train: 282 | Val: 32

Пример форматированной строки:
<|system|>You are a professional marketing copywriter. Write clear, concise, and compelling marketing content.</s><|user|>Generate a sales pitch for a marketing consultant.</s><|assistant|>Are you tired of your business lagging behind your competition? Do you want to see an increase in traffic, leads, and sales?  Then you need the help of a professional marketing consultant.

Our experienced consu


## Шаг 3- Загрузка модели в lora

Изначально планировала QLoRA (4-bit квантизация через bitsandbytes), но столкнулась с проблемой совместимости: Colab использует CUDA 13.0, а bitsandbytes на момент запуска не имел скомпилированного бинарника под эту версию. Все попытки установить совместимую версию (0.42.0, 0.43.3, 0.45.5) заканчивались ошибкой libbitsandbytes_cuda130.so not found.
Перешла на обычный LoRA в float16 модель загружается полностью в fp16, адаптеры обучаются поверх. Для TinyLlama-1.1B на T4 памяти хватает без квантизации (~2.2 GB модель + адаптеры).

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# QLoRA не работает на CUDA 13.0 bitsandbytes не поддерживает эту версию. Использую обычный LoRA в float16 результат сопоставимый, памяти хватает
# для TinyLlama на T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)

print('Модель загружена')
print(f'Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Модель загружена
Занято VRAM: 2.1 GB


## Шаг 4- LoRA конфигурация

r=16 rank адаптера, определяет сколько параметров обучается.Начала с r=8 loss падал медленно. При r=32 модель начинала переобучаться на val уже к концу 2-й эпохи. Взяла r=16 он оказался оптимальным для этого размера датасета.

target_modules слои attention куда вставляются адаптеры.
Сначала брала только q_proj и v_proj (как в оригинальном LoRA),
потом добавила k_proj и o_proj loss стал сходиться стабильнее.

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,            # alpha=2*r стандартное соотношение
    # target_modules=['q_proj', 'v_proj'],  # минимальный вариант
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# ожидаю~1-2% обучаемых параметров

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.40791125334440875


## Шаг 5 W&B логирование

In [8]:
import wandb
wandb.login(key="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")

wandb.init(
    project='marketing-finetune',
    name='tinyllama-qlora-r16',
    config={
        'model': MODEL_NAME,
        'dataset': 'alpaca-cleaned-marketing-640',
        'lora_r': 16,
        'lora_alpha': 32,
        'method': 'QLoRA 4-bit nf4',
        'epochs': 3,
        'batch_size': 4,
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: m-makpyr (m-makpyr-iitu-edu-kz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


batch_size=8 сразу падал с OOM, поэтому я сделала связку batch_size=4 + gradient_accumulation=4. Получился стабильный эффективный батч на 16.

Контекст (max_seq_length) ограничила в 512 токенов. Маркетинговые тексты из выборки короткие (обычно 200-300 токенов), поэтому 512 покрывает их полностью и не ест лишнюю память

In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./marketing-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=4,   # больше OOM на T4
    gradient_accumulation_steps=4,   # эффективный batch = 16
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='wandb',
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print('Начинаем обучение...')
print(f'Шагов всего: {len(dataset["train"]) // (4*4) * 3}')
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will over

Map:   0%|          | 0/282 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Начинаем обучение...
Шагов всего: 51


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
0,1.248400,0.970942
1,0.986900,0.953404
2,0.991800,0.951592


TrainOutput(global_step=51, training_loss=1.053528784536848, metrics={'train_runtime': 165.4161, 'train_samples_per_second': 5.114, 'train_steps_per_second': 0.308, 'total_flos': 2267158021447680.0, 'train_loss': 1.053528784536848, 'epoch': 2.873239436619718})

## Шаг 7 Сохранение адаптера

Сохраняю только адаптер (~10-30 MB), не всю модель (2.2 GB).
Для деплоя нужна будет базовая модель + адаптер отдельно.

In [10]:
import os

adapter_path = './marketing-lora-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

adapter_files = os.listdir(adapter_path)
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in adapter_files
) / 1024**2

print(f'Адаптер сохранён: {adapter_path}')
print(f'Файлы: {adapter_files}')
print(f'Размер адаптера: {total_size:.1f} MB')

wandb.finish()

Адаптер сохранён: ./marketing-lora-adapter
Файлы: ['tokenizer_config.json', 'special_tokens_map.json', 'README.md', 'tokenizer.model', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']
Размер адаптера: 21.1 MB


eval/loss,█▂▁
eval/runtime,█▁▃
eval/samples_per_second,▁█▆
eval/steps_per_second,▁█▆
train/epoch,▁▂▃▄▅▆███
train/global_step,▁▂▃▄▅▆███
train/grad_norm,█▁▂▂▁
train/learning_rate,█▆▄▂▁
train/loss,█▃▁▁▁
eval/loss,0.95159
eval/runtime,3.0112


## Шаг 8 Быстрый тест

Проверяю что модель вообще генерирует что-то осмысленное.Полноценная оценка будет в отдельном ноутбуке (evaluation.ipynb).

In [11]:
def generate(prompt, max_new_tokens=200):
    formatted = (
        f"<|system|>You are a professional marketing copywriter.</s>"
        f"<|user|>{prompt}</s>"
        f"<|assistant|>"
    )
    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('<|assistant|>')[-1].strip()

prompts = [
    'Write a catchy Instagram caption for a new coffee brand targeting young professionals.',
    'Create a slogan for an eco-friendly water bottle.',
    'Write a subject line for a promotional email about a 50% sale.',
]

for p in prompts:
    print(f'PROMPT: {p}')
    print(f'RESPONSE: {generate(p)}')
    print('-' * 60)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


PROMPT: Write a catchy Instagram caption for a new coffee brand targeting young professionals.
RESPONSE: Here's a catchy Instagram caption for a new coffee brand targeting young professionals: "Join the coffee revolution! 😍 Our fresh, flavorful coffee will set you free from the routine and bring you closer to the people and places that make your life amazing. Try our new line of specialty coffee blends and drink it up on your next break!"
------------------------------------------------------------
PROMPT: Create a slogan for an eco-friendly water bottle.
RESPONSE: Refill, Reuse, Reduce! With our eco-friendly water bottle, keep your daily hydration on track while minimizing waste and environmental impact.

Refill, Reuse, Reduce - These three words represent our commitment to sustainability, and our water bottle is designed to help you do just that. Whether you're a frequent traveler or a beach-goer, our bottle is made from materials that are both durable and eco-friendly. With every re

## Шаг 9 Скачать адаптер

In [12]:
import shutil
shutil.make_archive('marketing-lora-adapter', 'zip', adapter_path)

from google.colab import files
files.download('marketing-lora-adapter.zip')
print('Готово! Адаптер скачан.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Готово! Адаптер скачан.
